# Data Pipeline Portfolio Project

### **Objective**: build a data engineering pipeline to regularly transform a messy database into a clean source of truth for an analytics team.
### **Data Source:** Cademycode customer data
### **Data Format**: sqlite3 Database
#### **Author**: Zac Cheatle

In [1]:
# import dependencies
import os
import pandas as pd
import numpy as np
import json
import sqlite3
import logging

# set pandas column width
pd.set_option('display.max_colwidth', None)

In [2]:
# connect to sqlite3 db 

con = sqlite3.connect('cademycode.db')

curs = con.cursor()

In [3]:
# view tables

curs.execute('''SELECT name FROM sqlite_master WHERE type='table';''').fetchall()

[('cademycode_students',),
 ('cademycode_courses',),
 ('cademycode_student_jobs',)]

In [4]:
# import tables to dataframes (could do this with a loop but since only 3 tables going to do this 1 by 1)

# students
students = pd.read_sql_query('SELECT * FROM cademycode_students', con)
students.head(5)

,uuid,name,dob,sex,contact_info,job_id,num_course_taken,current_career_path_id,time_spent_hrs
0,1,Annabelle Avery,1943-07-03,F,"{""mailing_address"": ""303 N Timber Key, Irondale, Wisconsin, 84736"", ""email"": ""annabelle_avery9376@woohoo.com""}",7.0,6.0,1.0,4.99
1,2,Micah Rubio,1991-02-07,M,"{""mailing_address"": ""767 Crescent Fair, Shoals, Indiana, 37439"", ""email"": ""rubio6772@hmail.com""}",7.0,5.0,8.0,4.4
2,3,Hosea Dale,1989-12-07,M,"{""mailing_address"": ""P.O. Box 41269, St. Bonaventure, Virginia, 83637"", ""email"": ""hosea_dale8084@coldmail.com""}",7.0,8.0,8.0,6.74
3,4,Mariann Kirk,1988-07-31,F,"{""mailing_address"": ""517 SE Wintergreen Isle, Lane, Arkansas, 82242"", ""email"": ""kirk4005@hmail.com""}",6.0,7.0,9.0,12.31
4,5,Lucio Alexander,1963-08-31,M,"{""mailing_address"": ""18 Cinder Cliff, Doyles borough, Rhode Island, 73737"", ""email"": ""alexander9810@hmail.com""}",7.0,14.0,3.0,5.64


In [5]:
# import tables to dataframes

# courses
courses = pd.read_sql_query('SELECT * FROM cademycode_courses', con)
courses.head(5)

,career_path_id,career_path_name,hours_to_complete
0,1,data scientist,20
1,2,data engineer,20
2,3,data analyst,12
3,4,software engineering,25
4,5,backend engineer,18


In [6]:
# import tables to dataframes

# student jobs
student_jobs = pd.read_sql_query('SELECT * FROM cademycode_student_jobs', con)
student_jobs.head(5)

,job_id,job_category,avg_salary
0,1,analytics,86000
1,2,engineer,101000
2,3,software developer,110000
3,4,creative,66000
4,5,financial services,135000


## Data Prep

#### Shape

In [7]:
students.shape

(5000, 9)

In [8]:
courses.shape

(10, 3)

In [9]:
student_jobs.shape

(13, 3)

#### Column clean up

In [10]:
# split contact_info column into separate email and address columns
students['contact_info'] = students['contact_info'].apply(lambda x: json.loads(x) if isinstance(x, str) else x)
students['mailing_address'] = students['contact_info'].apply(lambda x: x.get('mailing_address') if isinstance(x, dict) else None)
students['email'] = students['contact_info'].apply(lambda x: x.get('email') if isinstance(x, dict) else None)

# split up mailing address into separate columns
students[['address', 'city', 'state', 'zip']] = students['mailing_address'].str.extract(r'^(.*),\s*([\w\s]+),\s*([\w\s]+),\s*(\d+)$')

In [11]:
# drop original contact_info column
students = students.drop(columns=['mailing_address'])
students = students.drop(columns=['contact_info'])
students.head()

,uuid,name,dob,sex,job_id,num_course_taken,current_career_path_id,time_spent_hrs,email,address,city,state,zip
0,1,Annabelle Avery,1943-07-03,F,7.0,6.0,1.0,4.99,annabelle_avery9376@woohoo.com,303 N Timber Key,Irondale,Wisconsin,84736
1,2,Micah Rubio,1991-02-07,M,7.0,5.0,8.0,4.4,rubio6772@hmail.com,767 Crescent Fair,Shoals,Indiana,37439
2,3,Hosea Dale,1989-12-07,M,7.0,8.0,8.0,6.74,hosea_dale8084@coldmail.com,NaN,NaN,NaN,NaN
3,4,Mariann Kirk,1988-07-31,F,6.0,7.0,9.0,12.31,kirk4005@hmail.com,517 SE Wintergreen Isle,Lane,Arkansas,82242
4,5,Lucio Alexander,1963-08-31,M,7.0,14.0,3.0,5.64,alexander9810@hmail.com,18 Cinder Cliff,Doyles borough,Rhode Island,73737


#### Duplicates

In [12]:
students.duplicated().value_counts()

False    5000
dtype: int64

In [13]:
courses.duplicated().value_counts()

False    10
dtype: int64

In [14]:
student_jobs.duplicated().value_counts()

False    10
True      3
dtype: int64

In [15]:
student_jobs=student_jobs.drop_duplicates(keep='last')

In [16]:
student_jobs.duplicated().value_counts()

False    10
dtype: int64

#### Missing Values

In [17]:
students.isnull().sum()

uuid                        0
name                        0
dob                         0
sex                         0
job_id                      5
num_course_taken          251
current_career_path_id    471
time_spent_hrs            471
email                       0
address                    77
city                       77
state                      77
zip                        77
dtype: int64

In [18]:
# drop all rows with a null value
students = students.dropna()

In [19]:
students.shape

(4224, 13)

In [20]:
courses.isnull().sum()

career_path_id       0
career_path_name     0
hours_to_complete    0
dtype: int64

In [21]:
student_jobs.isnull().sum()

job_id          0
job_category    0
avg_salary      0
dtype: int64

#### Data Types

In [22]:
students.dtypes

uuid                       int64
name                      object
dob                       object
sex                       object
job_id                    object
num_course_taken          object
current_career_path_id    object
time_spent_hrs            object
email                     object
address                   object
city                      object
state                     object
zip                       object
dtype: object

In [23]:
### set datatypes

dtype_dict = {'uuid':'int16',
              'name':'string', 
              'dob':'datetime64[ns]', 
              'sex':'category', 
              'job_id':'float16', 
              'num_course_taken':'float16', 
              'current_career_path_id':'float16', 
              'time_spent_hrs':'float16',
              'email':'string',
              'address':'string',
              'city':'string',
              'state':'category',
              'zip': 'string'}

students = students.astype(dtype_dict)
students.dtypes

uuid                               int16
name                              string
dob                       datetime64[ns]
sex                             category
job_id                           float16
num_course_taken                 float16
current_career_path_id           float16
time_spent_hrs                   float16
email                             string
address                           string
city                              string
state                           category
zip                               string
dtype: object

In [24]:
students['num_course_taken'] = students['num_course_taken'].astype(int)
students['job_id'] = students['job_id'].astype(int)
students['current_career_path_id'] = students['current_career_path_id'].astype(int)
students.head()

,uuid,name,dob,sex,job_id,num_course_taken,current_career_path_id,time_spent_hrs,email,address,city,state,zip
0,1,Annabelle Avery,1943-07-03,F,7,6,1,4.988281,annabelle_avery9376@woohoo.com,303 N Timber Key,Irondale,Wisconsin,84736
1,2,Micah Rubio,1991-02-07,M,7,5,8,4.398438,rubio6772@hmail.com,767 Crescent Fair,Shoals,Indiana,37439
3,4,Mariann Kirk,1988-07-31,F,6,7,9,12.312500,kirk4005@hmail.com,517 SE Wintergreen Isle,Lane,Arkansas,82242
4,5,Lucio Alexander,1963-08-31,M,7,14,3,5.640625,alexander9810@hmail.com,18 Cinder Cliff,Doyles borough,Rhode Island,73737
5,6,Shavonda Mcmahon,1989-10-15,F,6,10,3,10.117188,shavonda5863@coldmail.com,P.O. Box 81591,Tarpon Springs,Montana,37057


In [25]:
courses.dtypes

career_path_id        int64
career_path_name     object
hours_to_complete     int64
dtype: object

In [26]:
student_jobs.dtypes

job_id           int64
job_category    object
avg_salary       int64
dtype: object

## Export Cleaned Data

#### Create new sqlite3 database with cleaned tables

In [27]:
new_conn = sqlite3.connect('cleaned_cademycode.db')

students.to_sql('students', new_conn, index=False, if_exists='replace')
courses.to_sql('courses', new_conn, index=False, if_exists='replace')
student_jobs.to_sql('student_jobs', new_conn, index=False, if_exists='replace')

10

#### Create csv with all tables merged together for analysts

In [28]:
students_courses = students.merge(courses, left_on='current_career_path_id', right_on='career_path_id', how='left')
cleaned_cademycode_data = students_courses.merge(student_jobs, on='job_id', how='left')
cleaned_cademycode_data.drop(['career_path_id'], axis=1, inplace=True)
cleaned_cademycode_data.head()

,uuid,name,dob,sex,job_id,num_course_taken,current_career_path_id,time_spent_hrs,email,address,city,state,zip,career_path_name,hours_to_complete,job_category,avg_salary
0,1,Annabelle Avery,1943-07-03,F,7,6,1,4.988281,annabelle_avery9376@woohoo.com,303 N Timber Key,Irondale,Wisconsin,84736,data scientist,20,HR,80000
1,2,Micah Rubio,1991-02-07,M,7,5,8,4.398438,rubio6772@hmail.com,767 Crescent Fair,Shoals,Indiana,37439,android developer,27,HR,80000
2,4,Mariann Kirk,1988-07-31,F,6,7,9,12.312500,kirk4005@hmail.com,517 SE Wintergreen Isle,Lane,Arkansas,82242,machine learning engineer,35,education,61000
3,5,Lucio Alexander,1963-08-31,M,7,14,3,5.640625,alexander9810@hmail.com,18 Cinder Cliff,Doyles borough,Rhode Island,73737,data analyst,12,HR,80000
4,6,Shavonda Mcmahon,1989-10-15,F,6,10,3,10.117188,shavonda5863@coldmail.com,P.O. Box 81591,Tarpon Springs,Montana,37057,data analyst,12,education,61000


In [29]:
# create csv

cleaned_cademycode_data.to_csv('cleaned_cademycode_data', index=False)

In [30]:
#### commit and close db connections

con.close()

new_conn.commit()
new_conn.close()